# Option 1: One skill per employee - one skill per task

## 1. Problem Definition
In this initial phase, we are solving a static, continuous resource allocation problem. Our goal is to assign employee hours to specific project tasks over a 1-year period to minimize total payroll costs. 

To ensure realism, an employee can only be assigned to a task if their `Primary_Skill` matches the task's `Required_Skill`. 

## 2. Mathematical Formulation
Let us define the components of our Linear Programming (LP) model:

**Sets:**
* $E$: Set of all employees, indexed by $i$.
* $T$: Set of all project tasks, indexed by $j$.
* $V$: Set of all valid assignment pairs $(i,j)$ where employee $i$'s skill matches task $j$'s required skill.

**Parameters:**
* $c_i$: Hourly cost of employee $i$.
* $k_i$: Maximum annual available hours for employee $i$ (e.g., 2080).
* $d_j$: Total hours demanded by task $j$.

**Decision Variables:**
* $x_{ij}$: The continuous number of hours employee $i$ is assigned to task $j$. Defined only for $(i,j) \in V$.

**Objective Function:**
Minimize the total cost of task assignments:
$$Z = \sum_{(i,j) \in V} c_i \cdot x_{ij}$$

**Constraints:**
1. **Demand Satisfaction:** Every task must receive exactly the hours it requires.
$$\sum_{i \mid (i,j) \in V} x_{ij} = d_j \quad \forall j \in T$$

2. **Capacity Limit:** No employee can exceed their maximum available hours.
$$\sum_{j \mid (i,j) \in V} x_{ij} \le k_i \quad \forall i \in E$$

3. **Non-Negativity:** Hours assigned cannot be negative.
$$x_{ij} \ge 0 \quad \forall (i,j) \in V$$

In [2]:
import pandas as pd
from ortools.linear_solver import pywraplp

# 1. LOAD AND PREP THE SYNTHETIC DATA
try:
    employees_df = pd.read_csv('../dataset_exports/2026-03-31_18-23-44/employees.csv')
    tasks_df = pd.read_csv('../dataset_exports/2026-03-31_18-23-44/tasks.csv')
except FileNotFoundError:
    print("Error: Could not find the CSVs. Make sure you ran the generation script first!")
    raise

display(employees_df.head())
display(tasks_df.head())

# Convert the pipe-separated strings back into Python lists
employees_df['Categories'] = employees_df['Categories'].str.split('|')

# OR TRICK: Inject the External Contractor to guarantee feasibility
# We give them every category that exists in the task list
all_categories = list(tasks_df['Category_Needed'].unique())
contractor = pd.DataFrame([{
    'Employee_ID': 'EXT_CONTRACTOR',
    'Categories': all_categories,
    'Specific_Skills': 'ALL', # Ignored in this model
    'Hourly_Cost': 5000,
    'Max_Hours': 999999
}])
employees_df = pd.concat([employees_df, contractor], ignore_index=True)


# 2. GENERATE VALID PAIRS (CATEGORY LEVEL)
valid_pairs = []

for _, emp in employees_df.iterrows():
    for _, task in tasks_df.iterrows():
        # THE CORE LOGIC: Does the employee have the Category the task needs?
        if task['Category_Needed'] in emp['Categories'][0]:
            valid_pairs.append((emp['Employee_ID'], task['Task_ID']))

print(f"Generated {len(valid_pairs)} valid (Employee, Task) assignment pairs based on Categories.")


# 3. BUILD AND RUN THE OR-TOOLS SOLVER
# Initialize the Continuous Linear Programming solver (GLOP)
solver = pywraplp.Solver.CreateSolver('GLOP')

# A. Decision Variables: x_ij (Continuous hours)
x = {}
for (i, j) in valid_pairs:
    x[(i, j)] = solver.NumVar(0, solver.infinity(), f'x_{i}_{j}')

# B. Objective Function: Minimize Cost
cost_dict = employees_df.set_index('Employee_ID')['Hourly_Cost'].to_dict()
objective = solver.Objective()
for (i, j) in valid_pairs:
    objective.SetCoefficient(x[(i, j)], float(cost_dict[i]))
objective.SetMinimization()

# C. Constraint 1: Demand Satisfaction (Task needs X hours)
demand_dict = tasks_df.set_index('Task_ID')['Hours_Needed'].to_dict()
for j in tasks_df['Task_ID']:
    capable_employees = [i for (i, t_id) in valid_pairs if t_id == j]
    constraint = solver.Constraint(float(demand_dict[j]), float(demand_dict[j]), f"Demand_{j}")
    for i in capable_employees:
        constraint.SetCoefficient(x[(i, j)], 1)

# D. Constraint 2: Capacity Limit (Employee has max 2080 hours)
cap_dict = employees_df.set_index('Employee_ID')['Max_Hours'].to_dict()
for i in employees_df['Employee_ID']:
    assigned_tasks = [j for (e_id, j) in valid_pairs if e_id == i]
    constraint = solver.Constraint(0, float(cap_dict[i]), f"Capacity_{i}")
    for j in assigned_tasks:
        constraint.SetCoefficient(x[(i, j)], 1)

status = solver.Solve()

# 4. EXTRACT AND DISPLAY RESULTS
if status == pywraplp.Solver.OPTIMAL:
    print(f"\n✅ Status: OPTIMAL")
    print(f"💰 Total Baseline Cost: ${solver.Objective().Value():,.2f}")
    
    # Extract non-zero assignments into a list
    results = []
    for (i, j) in valid_pairs:
        assigned_hours = x[(i, j)].solution_value()
        if assigned_hours > 0: 
            results.append({
                'Task_ID': j,
                'Employee_ID': i,
                'Assigned_Hours': assigned_hours,
                'Cost': assigned_hours * cost_dict[i]
            })
            
    results_df = pd.DataFrame(results).sort_values(by=['Task_ID', 'Employee_ID']).reset_index(drop=True)
    
    # Merge with tasks_df to show the Category for the presentation
    display_df = results_df.merge(tasks_df[['Task_ID', 'Category_Needed']], on='Task_ID')
    
    print("\n--- Snippet of Optimal Assignment Schedule ---")
    display(display_df.head(15))
    
    # Check if the external contractor was used
    contractor_hours = results_df[results_df['Employee_ID'] == 'EXT_CONTRACTOR']['Assigned_Hours'].sum()
    if contractor_hours > 0:
        print(f"\n🚨 ALERT: Internal capacity exceeded. Outsourced {contractor_hours:,.0f} hours to EXT_CONTRACTOR.")
    else:
        print("\n🎉 SUCCESS: 100% of demand met by internal staff.")

else:
    print("Solver failed. Status:", status)

,Employee_ID,Categories,Specific_Skills,Hourly_Cost,Max_Hours
0,E001,Backend|Testing,Go|PyTest,67,2080
1,E002,Data|Frontend,SQL|Tableau|React,86,2080
2,E003,Data,Pandas|Tableau,65,2080
3,E004,Frontend|DevOps,TypeScript|Angular|Vue.js|AWS,97,2080
4,E005,DevOps,Kubernetes,58,2080


,Project_ID,Task_ID,Category_Needed,Skills_Needed,Hours_Needed
0,P001,P001_T1,Testing,JUnit,400
1,P001,P001_T2,Backend,C#|Node.js,600
2,P002,P002_T1,Data,Spark|Tableau,800
3,P002,P002_T2,Data,Tableau|SQL,200
4,P003,P003_T1,Backend,Java|Python,300


Generated 476 valid (Employee, Task) assignment pairs based on Categories.

✅ Status: OPTIMAL
💰 Total Baseline Cost: $950,500.00

--- Snippet of Optimal Assignment Schedule ---


,Task_ID,Employee_ID,Assigned_Hours,Cost,Category_Needed
0,P001_T1,E009,400.0,20800.0,Testing
1,P001_T2,E007,220.0,11440.0,Backend
2,P001_T2,E048,380.0,19380.0,Backend
3,P002_T1,E040,800.0,48000.0,Data
4,P002_T2,E016,200.0,12800.0,Data
5,P003_T1,E048,300.0,15300.0,Backend
6,P003_T2,E016,600.0,38400.0,Data
7,P004_T1,E009,200.0,10400.0,Testing
8,P004_T2,E007,200.0,10400.0,Backend
9,P004_T3,E057,600.0,37200.0,Frontend



🎉 SUCCESS: 100% of demand met by internal staff.
